# Extract somatic dynamics of human cells from Allen in vitro recordings

In [1]:
import os
import os.path as op

from allensdk.core.cell_types_cache import CellTypesCache
from allensdk.ephys.extract_cell_features import extract_cell_features

import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
import pickle

# define paths
path_name = '/Users/katharinaduecker/Documents/projects_brown/hnn-tuning/'
path_data = op.join(path_name, 'data', 'allen-data')



In [2]:
# get CSV with specimen details
df = pd.read_csv(op.join(path_data, 'cell_types_specimen_details.csv'))
# get cell types cache
ctc = CellTypesCache(manifest_file='cell_types/manifest.json')


In [3]:
# exponential function to fit membrane time constant
def exp_func_mem_time(t, tau, A, C):
    return A * (1 - np.exp(-t / tau)) + C

In [4]:
# algorithm to extract features of interest 
def extract_ephys(specimen_ids): 

    ephys_features = dict()

    for i, specimen_id in enumerate(specimen_ids.values):

        # download the ephys data and sweep metadata
        data_set = ctc.get_ephys_data(specimen_id)
        sweeps = ctc.get_ephys_sweeps(specimen_id)

        # group the sweeps by stimulus 
        sweep_numbers = defaultdict(list)
        for sweep in sweeps:
            sweep_numbers[sweep['stimulus_name']].append(sweep['sweep_number'])

        # calculate features

        ephys_features[specimen_id] = dict()
        ephys_features[specimen_id]['index'] = specimen_id

        try: 
            cell_features = extract_cell_features(data_set,
                                                sweep_numbers['Ramp'],
                                                sweep_numbers['Short Square'],
                                                sweep_numbers['Long Square'])
            # calculate features
            long_square = cell_features['long_squares']

            # Assuming your dictionary is named 'data'
            long_df = pd.DataFrame(long_square['sweeps'])

            # mebrane potential
            ephys_features[specimen_id]['RMP'] = np.mean(long_df.v_baseline.values)

            # sag
            ephys_features[specimen_id]['sag'] = np.mean(long_df['sag'][long_df['stim_amp'] < 0].values)

            
            # get amplitudes used to extract membrane time constant tau
            ephys_features[specimen_id]['sag_stim_amp'] = long_df['stim_amp'][long_df['stim_amp'] < 0].values/1000

            # get ISIs for stimuli with positive amplitude (depol.)
            ephys_features[specimen_id]['max_amp_isi'] = np.max(long_df['stim_amp'])/1000

            # get action potential features
            short_square = cell_features['short_squares']
            ephys_features[specimen_id]['upstroke_downstroke'] = short_square['common_amp_sweeps'][0]['spikes'][0]['upstroke_downstroke_ratio']
            ephys_features[specimen_id]['ap_width'] = short_square['common_amp_sweeps'][0]['spikes'][0]['width']*1000
            ephys_features[specimen_id]['ap_threshold'] = short_square['common_amp_sweeps'][0]['spikes'][0]['threshold_v']

            sweeps = data_set.get_experiment_sweep_numbers()
            sweeps_meta = dict()
            for sweep in sweeps:
                sweeps_meta[sweep] = data_set.get_sweep_metadata(sweep)

            sweeps_df = pd.DataFrame.from_dict(sweeps_meta, orient='index')

            # extract long voltage sweeeps
            sweeps_long = sweeps_df[(sweeps_df['aibs_stimulus_name'] == b'Long Square') & (sweeps_df['aibs_stimulus_amplitude_pa'] >0)]

            # number of spikes
            num_spks = np.zeros_like(sweeps_long.index)
            # amplitude of current injection
            ampl = np.zeros_like(sweeps_long.index)
            # ISI of first spikes
            first_isi = np.zeros_like(sweeps_long.index)
            # ISI of following spikes
            mean_isi = np.zeros_like(sweeps_long.index)

            for i,s in enumerate(sweeps_long.index):
                sweep = data_set.get_sweep(s)

                # get firing rate slope
                v = sweep['response']
                pk = find_peaks(v,height=0)
                num_spks[i] = len(pk[0])
                ampl[i] = sweeps_long['aibs_stimulus_amplitude_pa'][s]

                dt = 1.0e3 / sweep['sampling_rate']
                num_samples = len(v)
                t = np.arange(num_samples) * dt

                # find ISI because I'm unsure about the units used by Allen
                if len(pk[0]) > 1:
                    num_spks[i] = len(pk[0])
                    ampl[i] = sweeps_long['aibs_stimulus_amplitude_pa'][s]

                    dt = 1.0e3 / sweep['sampling_rate']
                    num_samples = len(v)
                    t = np.arange(num_samples) * dt

                    first_isi[i] = np.diff(t[pk[0]])[0]
                    mean_isi[i] = np.mean(np.diff(t[pk[0]]))
            
            ephys_features[specimen_id]['FR_slope'] = np.polyfit(ampl[num_spks >0],num_spks[num_spks >0],1)[0]
            ephys_features[specimen_id]['first_isi'] = np.median(first_isi[first_isi>0])                    # to get a more representative feature, take the median over all sweeps
            ephys_features[specimen_id]['mean_isi'] = np.median(first_isi[mean_isi>0])

            # mebrane time constant based on hyperpolarization experiments
            sweeps_long = sweeps_df[(sweeps_df['aibs_stimulus_name'] == b'Long Square') & (sweeps_df['aibs_stimulus_amplitude_pa'] < 0)]
            tau_ms = np.zeros_like(sweeps_long.index)
            for i,s in enumerate(sweeps_long.index):
                sweep = data_set.get_sweep(s)
                
                # get membrane time constant
                t_max = len(sweep['response'])/sweep['sampling_rate']*1000          # convert seconds to ms
                timevec = np.linspace(0, t_max, len(sweep['response']))

                stim_onset = np.where(sweep['stimulus']<0)[0][0]
                stim_offset = np.where(sweep['stimulus'][stim_onset:]>=0)[0][0]+stim_onset

                t_rise = timevec[stim_onset:stim_offset]-timevec[stim_onset]
                v_rise = sweep['response'][stim_onset:stim_offset]
                # baseline correction
                delta_v = np.mean(v_rise) - np.mean(sweep['response'][:stim_onset])
                popt, pcov = curve_fit(exp_func_mem_time, t_rise, v_rise, p0=[10, abs(delta_v), v_rise[0]])
                tau_ms[i] = popt[0]
                        
            # tau
            ephys_features[specimen_id]['tau'] = np.mean(tau_ms)
            
            del sweep, v, ampl, short_square, long_square, long_df, data_set

        except:

            ephys_features[specimen_id]['RMP'] = float('nan')
            ephys_features[specimen_id]['sag'] = float('nan')
            ephys_features[specimen_id]['mean_isi'] = float('nan')
            ephys_features[specimen_id]['first_isi'] = float('nan')
            ephys_features[specimen_id]['upstroke_downstroke'] = float('nan')
            ephys_features[specimen_id]['ap_width'] = float('nan')
            ephys_features[specimen_id]['ap_threshold'] = float('nan')
            ephys_features[specimen_id]['FR_slope'] = float('nan')
            ephys_features[specimen_id]['tau'] = float('nan')

    return ephys_features


In [5]:
# specimen_ids = df['specimen__id'][(df['donor__species'] == 'Homo Sapiens') & (df['tag__dendrite_type'] == 'spiny') & (df['structure__layer'] == '2')]
# specimen_id = specimen_ids.iloc[0]

# # download the ephys data and sweep metadata
# data_set = ctc.get_ephys_data(specimen_id)
# sweeps = ctc.get_ephys_sweeps(specimen_id)

# # group the sweeps by stimulus 
# sweep_numbers = defaultdict(list)
# for sweep in sweeps:
#     sweep_numbers[sweep['stimulus_name']].append(sweep['sweep_number'])

# # calculate features

# cell_features = extract_cell_features(data_set,
#                                     sweep_numbers['Ramp'],
#                                     sweep_numbers['Short Square'],
#                                     sweep_numbers['Long Square'])
# # calculate features
# long_square = cell_features['long_squares']

# # Assuming your dictionary is named 'data'
# long_df = pd.DataFrame(long_square['sweeps'])

# # Set 'id' as the index
# long_df.set_index('id', inplace=True)

In [18]:
def compare_df(old, new):
    try:
        pd.testing.assert_frame_equal(old, new, rtol=1e-12, atol=1e-9)
        print(f"MATCH specimens")
    except AssertionError as exc:
        print(f"DIFFERS")
        print("   ", str(exc).splitlines()[0])

In [23]:

with open(os.path.join(path_data,'interneuron','ephys_inh.pkl'), 'rb') as file:
    ephys_inh = pickle.load(file)

with open(os.path.join(path_data,'interneuron','ephys_inh_new.pkl'), 'rb') as file:
    ephys_inh_new = pickle.load(file)

In [24]:
compare_df(ephys_inh, ephys_inh_new)

MATCH specimens


In [25]:
ephys_inh.equals(ephys_inh_new)

True

In [21]:
with open(os.path.join(path_data,'l23','ephys_l2.pkl'), 'rb') as file:
    ephys_l2 = pickle.load(file)

with open(os.path.join(path_data,'l23','ephys_l2_new.pkl'), 'rb') as file:
    ephys_l2_new = pickle.load(file)


compare_df(ephys_l2, ephys_l2_new)

MATCH specimens


In [22]:
ephys_l2.equals(ephys_l2_new) 

True

In [37]:
np.where(np.isnan(ephys_inh["RMP"]))

(array([17]),)

In [26]:
with open(os.path.join(path_data,'l23','ephys_l3.pkl'), 'rb') as file:
    ephys_l3 = pickle.load(file)

with open(os.path.join(path_data,'l23','ephys_l3_new.pkl'), 'rb') as file:
    ephys_l3_new = pickle.load(file)

In [40]:
ephys_l3.equals(ephys_l3_new)

True

In [39]:
compare_df(ephys_l3, ephys_l3_new)

MATCH specimens


In [29]:
ephys_l3

,index,RMP,sag,mean_isi,first_isi,upstroke_downstroke,ap_width,ap_threshold,FR_slope,tau,sag_stim_amp,max_amp_isi
525011903,525011903,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
519832676,519832676,-72.547661,0.198682,66.0,66.0,5.010128,1.14,-51.062504,0.056507,31.666667,"[-0.11, -0.029999997, -0.01]",0.23
545608578,545608578,-74.496262,0.100030,8.0,8.0,3.606008,0.94,-50.718750,0.053792,9.200000,"[-0.11, -0.09, -0.07, -0.05, -0.029999997]",0.71
561467633,561467633,-69.626610,0.277350,18.0,18.0,3.979118,0.80,-53.375004,0.030701,14.333333,"[-0.11, -0.09, -0.07, -0.05, -0.029999997, -0.01]",0.29
530055374,530055374,-74.551086,0.155382,35.5,35.5,3.452404,0.92,-47.656254,0.008000,20.500000,"[-0.11, -0.09, -0.07, -0.05, -0.029999997, -0.01]",0.39
...,...,...,...,...,...,...,...,...,...,...,...,...
528625591,528625591,-73.645546,0.120701,42.0,42.0,3.468519,0.88,-46.875004,0.017267,11.833333,"[-0.11, -0.09, -0.07, -0.05, -0.029999997, -0.01]",0.45
505690787,505690787,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
517689107,517689107,-71.924828,0.206797,6.5,6.5,2.977585,0.76,-37.125004,0.007674,6.600000,"[-0.11, -0.09, -0.07, -0.05, -0.029999997]",0.47
508298270,508298270,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
ephys_l3_new

,index,RMP,sag,sag_stim_amp,max_amp_isi,upstroke_downstroke,ap_width,ap_threshold,FR_slope,first_isi,mean_isi,tau
525011903,525011903,-70.561035,0.113496,"[-0.11, -0.09, -0.07, -0.05, -0.029999997, -0.01]",0.23,3.248495,0.80,-50.375004,0.167057,53.0,53.0,14.166667
519832676,519832676,-72.547661,0.198682,"[-0.11, -0.029999997, -0.01]",0.23,5.010128,1.14,-51.062504,0.056507,66.0,66.0,31.666667
545608578,545608578,-74.496262,0.100030,"[-0.11, -0.09, -0.07, -0.05, -0.029999997]",0.71,3.606008,0.94,-50.718750,0.053792,8.0,8.0,9.200000
561467633,561467633,-69.626610,0.277350,"[-0.11, -0.09, -0.07, -0.05, -0.029999997, -0.01]",0.29,3.979118,0.80,-53.375004,0.030701,18.0,18.0,14.333333
530055374,530055374,-74.551086,0.155382,"[-0.11, -0.09, -0.07, -0.05, -0.029999997, -0.01]",0.39,3.452404,0.92,-47.656254,0.008000,35.5,35.5,20.500000
...,...,...,...,...,...,...,...,...,...,...,...,...
528625591,528625591,-73.645546,0.120701,"[-0.11, -0.09, -0.07, -0.05, -0.029999997, -0.01]",0.45,3.468519,0.88,-46.875004,0.017267,42.0,42.0,11.833333
505690787,505690787,-71.074661,0.089762,"[-0.11, -0.09, -0.07, -0.05, -0.029999997, -0.01]",0.31,3.542487,0.82,-46.562504,0.088498,80.0,80.0,21.500000
517689107,517689107,-71.924828,0.206797,"[-0.11, -0.09, -0.07, -0.05, -0.029999997]",0.47,2.977585,0.76,-37.125004,0.007674,6.5,6.5,6.600000
508298270,508298270,-74.491547,0.100440,"[-0.11, -0.09, -0.07, -0.05, -0.029999997, -0.01]",0.35,2.398959,0.84,-49.500004,0.009036,7.0,7.0,27.166667


## Interneurons (human aspiny cells)

In [25]:

# create directory that summarizes features 
os.chdir(os.path.join(path_data,'interneuron'))

# inhibitory
specimen_ids = df['specimen__id'][(df['donor__species'] == 'Homo Sapiens') & (df['tag__dendrite_type'] == 'aspiny')]
ephys_features = extract_ephys(specimen_ids)
ephys_inh = pd.DataFrame.from_dict(ephys_features,orient='index')
ephys_inh.to_csv(os.path.join(path_data,'interneuron','ephys_inh.csv'))
ephys_inh.to_pickle(os.path.join(path_data,'interneuron','ephys_inh.pkl'))

/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, y)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1134: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, v)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean o

## Pyramidal neurons (human spiny cells in L2 and L3)

### L2 cells

In [34]:
specimen_ids = df['specimen__id'][(df['donor__species'] == 'Homo Sapiens') & (df['tag__dendrite_type'] == 'spiny') & (df['structure__layer'] == '2')]
ephys_features = extract_ephys(specimen_ids)

ephys_l2 = pd.DataFrame.from_dict(ephys_features,orient='index')
ephys_l2.to_csv(os.path.join(path_data,'l23','ephys_l2.csv'))
ephys_l2.to_pickle(os.path.join(path_data,'l23','ephys_l2.pkl'))

/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, y)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1134: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, v)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning:

### L3 cells

In [38]:
specimen_ids = df['specimen__id'][(df['donor__species'] == 'Homo Sapiens') & (df['tag__dendrite_type'] == 'spiny') & (df['structure__layer'] == '3')]

ephys_features = extract_ephys(specimen_ids)

ephys_l3 = pd.DataFrame.from_dict(ephys_features,orient='index')
ephys_l3.to_csv(os.path.join(path_data,'l23','ephys_l3.csv'))
ephys_l3.to_pickle(os.path.join(path_data,'l23','ephys_l3.pkl'))

/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, y)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1134: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, c = np.linalg.lstsq(A, v)[0]
/Users/katharinaduecker/miniconda3/envs/allensdk/lib/python3.11/site-packages/allensdk/ephys/ephys_extractor.py:1163: FutureWarning: